In [1]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

In [2]:
DATA_PATH = "m5-forecasting-uncertainty"

calendar = pd.read_csv(f"{DATA_PATH}/calendar.csv")
prices = pd.read_csv(f"{DATA_PATH}/sell_prices.csv")
sales = pd.read_csv(f"{DATA_PATH}/sales_train_evaluation.csv")

In [3]:
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
value_cols = [col for col in sales.columns if col.startswith("d_")]

df = sales.melt(id_vars=id_cols, value_vars=value_cols,
                var_name="d", value_name="sales")

In [4]:
df = df.merge(calendar, on="d", how="left")
df = df.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["id", "date"])

In [5]:
for lag in [1, 7, 14, 28]:
    df[f"lag_{lag}"] = df.groupby("id")["sales"].shift(lag)

df["rmean_7"] = df.groupby("id")["sales"].shift(1).rolling(7).mean()
df["rmean_28"] = df.groupby("id")["sales"].shift(1).rolling(28).mean()

df["weekday"] = df["date"].dt.weekday
df["month"] = df["date"].dt.month

df = df.dropna()

In [6]:
features = [
    "lag_1", "lag_7", "lag_14", "lag_28",
    "rmean_7", "rmean_28",
    "sell_price",
    "weekday", "month"
]

X = df[features]
y = df["sales"]

In [7]:
df["date"].max()

Timestamp('2014-06-15 00:00:00')

In [8]:
split_date = "2014-06-01"

X_train = X[df["date"] < split_date]
y_train = y[df["date"] < split_date]

In [9]:
quantiles = [0.1, 0.5, 0.9]  # you can expand to full Kaggle set
models = {}

for q in quantiles:
    print(f"Training quantile {q}")
    
    model = HistGradientBoostingRegressor(
        loss="quantile",
        quantile=q,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    models[q] = model

Training quantile 0.1
Training quantile 0.5
Training quantile 0.9


In [10]:
predictions = {}

for q, model in models.items():
    predictions[q] = model.predict(X_train.tail(1000))  # example

# Convert to DataFrame
pred_df = pd.DataFrame(predictions)
print(pred_df.head())

   0.1     0.5       0.9
0  0.0  0.0000  1.126262
1  0.0  0.0000  1.852026
2  0.0  0.0000  0.982692
3  0.0  0.9997  2.839326
4  0.0  0.0000  0.879578
